<a href="https://colab.research.google.com/github/EnzoSamp/COS360-Projeto-de-Otimizacao/blob/main/investimentoOTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introdução
**Esse notebook não serve como um relatório, tem apenas como objetivo mostrar os código dos modelo matemáticos.**

A otimização de carteiras de investimento é um pilar central da moderna teoria financeira, iniciada com o trabalho de Markowitz sobre mean-variance na década de 1950. Desde então, diversas métricas e modelos foram desenvolvidos para acomodar diferentes preferências de risco, como o CVaR, o qual foca nas perdas extremas, e índices de desempenho que medem relação entre ganhos e perdas, como o Omega Ratio. Esses modelos são amplamente empregados por gestoras, fundos institucionais e plataformas automatizadas para construir carteiras que respeitem restrições práticas (limites setoriais, limites por ativo, restrições de cardinalidade) e objetivos financeiros (maximizar retorno líquido, controlar risco de cauda).

Este trabalho implementa e compara três abordagens de otimização de portfólios que representam perfis distintos de investidor:
1. **Conservador:** Minimização do CVaR;
2. **Moderado:** Markowitz com ajuste de aversão ao risco (MIQP);
3. **Agressivo:** Maximização do Omega Ratio usando a transformação de Charnes-Cooper.

As implementações incorporam restrições realistas como cardinalidade (seleção limitada de ativos), limites setoriais e semicontinuidade por ativo (para forçar pesos mínimos quando selecionados), além de custos transacionais aproximados usados para computar retornos líquidos.

A motivação do trabalho consiste em fazer uma carteira de investimento para o integrante do grupo Enzo o qual apenas investe na SELIC atualmente. Desse modo, analisando 3 perfis gostaríamos de saber qual perfil de investidor se adequaria mais a ele, e a partir disso, criar uma carteira de investimento do zero com base no estudo dos 3 modelos.

In [ ]:
# @title Instalando bibliotecas
!pip install yfinance
!pip install pandas
!pip install python-bcb
!pip install gurobipy

In [ ]:
# @title Importando bibliotecas
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import gurobipy as gp
import numpy as np
from gurobipy import GRB
import math
import datetime
from bcb import sgs
import matplotlib.pyplot as plt

In [ ]:
# @title Licença do Gurobi
# Defina seus parâmetros de licença aqui
# (Substitua pelos valores que você pegou no portal/arquivo de licença)
params = {
    "WLSACCESSID": "",
    "WLSSECRET": "",
    "LICENSEID":   # Coloque o ID numérico da sua licença
}

env = gp.Env(params=params)
#Para rodar o código precisa da licença do Gurobi

## Tratamento dos dados dos ativos


In [ ]:
# @title Função geradora da lista de tickers
quantidade_de_anos = 3 # colocando quantos anos de dados você quer
def gerar_lista_simples():
    # 1. Ações Brasileiras (Setores diversos: Bancos, Energia, Varejo, Commodities)
    # Adicionamos o sufixo .SA para o yfinance -> SA (South America)
    acoes_br = [
        'ABEV3.SA', 'ALPA4.SA', 'AMAR3.SA', 'ANIM3.SA', 'AURE3.SA', 'AZZA3.SA', 'AZUL4.SA', 'B3SA3.SA', 'BBAS3.SA',
        'BBDC3.SA', 'BBDC4.SA', 'BBSE3.SA', 'BEEF3.SA', 'BPAC11.SA','BRAP4.SA', 'BRKM5.SA',
        'BRSR6.SA', 'CAML3.SA', 'CMIG4.SA', 'COGN3.SA', 'CPFE3.SA',
        'CPLE6.SA', 'CSAN3.SA', 'CSMG3.SA', 'CSNA3.SA', 'CVCB3.SA', 'CYRE3.SA', 'DIRR3.SA', 'DXCO3.SA',
        'ECOR3.SA', 'EGIE3.SA', 'ELET3.SA', 'ELET6.SA', 'EMBR3.SA', 'ENEV3.SA', 'ENGI11.SA',
        'EQTL3.SA', 'EVEN3.SA', 'EZTC3.SA', 'FESA4.SA', 'FLRY3.SA', 'GOLL54.SA', 'GFSA3.SA', 'GGBR4.SA',
        'GOAU4.SA', 'GRND3.SA', 'GUAR3.SA', 'HBOR3.SA', 'HYPE3.SA', 'IRBR3.SA',
        'ITSA4.SA', 'ITUB3.SA', 'ITUB4.SA', 'JBSS32.SA', 'JHSF3.SA', 'KLBN11.SA', 'LEVE3.SA', 'LIGT3.SA',
        'LOGG3.SA', 'LREN3.SA', 'MDIA3.SA', 'MEAL3.SA', 'MGLU3.SA', 'MILS3.SA', 'MOVI3.SA',
        'MRVE3.SA', 'MULT3.SA', 'MYPK3.SA', 'ODPV3.SA', 'PETR3.SA', 'PETR4.SA', 'POMO4.SA', 'PRIO3.SA',
        'PSSA3.SA', 'PTBL3.SA', 'QUAL3.SA', 'RADL3.SA', 'RAIL3.SA', 'RANI3.SA', 'RAPT4.SA', 'RENT3.SA', 'RDOR3.SA',
        'ROMI3.SA', 'SANB11.SA', 'SAPR11.SA', 'SAPR4.SA', 'SBSP3.SA', 'SEER3.SA', 'SLCE3.SA',
        'SMTO3.SA', 'SUZB3.SA', 'TAEE11.SA', 'TASA4.SA', 'TGMA3.SA', 'TIMS3.SA',
        'TOTS3.SA', 'TRIS3.SA', 'TUPY3.SA', 'UGPA3.SA', 'UNIP6.SA', 'USIM5.SA', 'VALE3.SA',
        'VIVT3.SA', 'VLID3.SA', 'VULC3.SA', 'WEGE3.SA', 'WIZC3.SA', 'YDUQ3.SA', 'AALR3.SA', 'AGRO3.SA',
        'BMEB4.SA', 'DXCO3.SA', 'POSI3.SA', 'SHUL4.SA', 'TEND3.SA', 'TRAD3.SA'
    ]

    # 2. Fundos Imobiliários
    fiis = [
        'ABCP11.SA', 'BBPO11.SA', 'BRCO11.SA', 'BRCR11.SA', 'CARE11.SA', 'CBOP11.SA',
        'CEOC11.SA', 'CNES11.SA', 'EDGA11.SA', 'FAED11.SA', 'FAMB11.SA', 'FCFL11.SA', 'FIGS11.SA',
        'FLMA11.SA', 'GGRC11.SA', 'HCTR11.SA', 'HFOF11.SA', 'HGBS11.SA', 'HGCR11.SA', 'HGLG11.SA',
        'HGRE11.SA', 'HGRU11.SA', 'HTMX11.SA', 'IGTI11.SA', 'KNCR11.SA', 'KNIP11.SA', 'KNRI11.SA',
        'MBRF11.SA', 'MFII11.SA', 'MXRF11.SA', 'NSLU11.SA', 'OUJP11.SA', 'PATL11.SA',
        'PLCR11.SA', 'PQDP11.SA', 'RBRF11.SA', 'RBRP11.SA', 'RBVA11.SA', 'RECT11.SA', 'RNGO11.SA',
        'SAAG11.SA', 'SDIL11.SA', 'SPTW11.SA', 'TGAR11.SA', 'TRXF11.SA', 'UBSR11.SA', 'VCJR11.SA',
        'VISC11.SA', 'VINO11.SA', 'VRTA11.SA', 'XPCM11.SA', 'XPIN11.SA', 'XPML11.SA'
    ]

    # 3. ETFs Brasileiros
    etfs = ['BOVA11.SA', 'SMAL11.SA', 'IVVB11.SA', 'DIVO11.SA', 'FIND11.SA', 'GOVE11.SA', 'MATB11.SA', 'PIBB11.SA']

    # 4. Ações Globais e América do Sul
    global_assets = [
        # LatAm Giants
        'MELI', 'NU', 'PAGS', 'STNE', 'ERJ', 'PBR', 'VALE', 'ITUB', 'BBD', 'BSAC', 'SQM', 'CIB', 'BAP',
        # Tech Giants
        'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'AMD', 'NFLX', 'INTC', 'CRM', 'ADBE', 'PYPL',
        # Finance & Payments
        'JPM', 'BAC', 'V', 'MA', 'GS', 'MS', 'WFC', 'C', 'AXP', 'BLK',
        # Consumer & Retail
        'PG', 'KO', 'PEP', 'COST', 'WMT', 'MCD', 'NKE', 'SBUX', 'DIS', 'HD', 'LOW', 'TGT',
        # Pharma & Health
        'JNJ', 'PFE', 'LLY', 'UNH', 'ABBV', 'MRK', 'TMO', 'DHR',
        # Industrial & Energy
        'XOM', 'CVX', 'GE', 'CAT', 'BA', 'LMT', 'RTX', 'UPS', 'FDX', 'MMM', 'HON',
        # Outros (Commodities, Chips, etc.)
        'TSM', 'ASML', 'AVGO', 'QCOM', 'TXN', 'ORCL', 'IBM', 'CSCO', 'ACN', 'LIN', 'RIO', 'BHP'
    ]

    sp100_extra = [
        'AMT', 'AMGN', 'AIG', 'ALL', 'MO', 'AEP', 'APA', 'BK', 'BAX', 'BMY', 'BIIB',
        'COF', 'CL', 'CMCSA', 'COP', 'CVS', 'D', 'DOW', 'DUK', 'EMR', 'EXC', 'F', 'GD',
        'GILD', 'GM', 'HAL', 'HPQ', 'KMI', 'KHC', 'KMB', 'K', 'LUV', 'MAR', 'MMC', 'MDLZ',
        'MET', 'MSI', 'NEE', 'NEM', 'OXY', 'ORLY', 'PRU', 'QCOM', 'REGN', 'ROST', 'SLB',
        'SO', 'SPG', 'T', 'TJX', 'USB', 'VZ', 'WMB'
    ]

    cripto = ['BTC-USD', 'ETH-USD']

    todos_tickers = acoes_br + fiis + etfs + global_assets + sp100_extra + cripto

    # Remover duplicatas
    todos_tickers = list(set(todos_tickers))

    print(f"Total de ativos na lista candidata: {len(todos_tickers)}")

    # Validação de Data
    fim = datetime.date.today()
    inicio = fim - datetime.timedelta(days=365*quantidade_de_anos + 20)

    ativos_validos = []

    print("Baixando e validando dados...")
    dados = yf.download(todos_tickers, start=inicio, end=fim, group_by='ticker', progress=True)

    for ticker in todos_tickers:
        try:
            df = dados[ticker]
            # Critérios para a validação:
            # 1. Não pode estar vazio
            # 2. Primeira data tem que ser próxima do inicio desejado (a gente vai colocar uma tolerancia de 30 dias)
            # 3. Ter uma quantidade decente de linhas (>1000, queremos grande volume de dados)
            if not df.empty:
                primeira_data = df.index[0].date()
                if primeira_data <= (inicio + datetime.timedelta(days=60)) and len(df) > 1000:
                    ativos_validos.append(ticker)
        except Exception:
            continue

    print(f"\nAtivos finais validados: {len(ativos_validos)}")

    return ativos_validos

In [ ]:
# @title Função para obter os dados dos tickers
def obter_dados_final(lista_ativos, anos_historico=quantidade_de_anos):

    tickers_download = list(set(lista_ativos))

    # Colocando o dólar na lista
    tem_internacional = any(not t.endswith('.SA') and t != 'SELIC_SINTETICA' for t in tickers_download)
    if tem_internacional and 'BRL=X' not in tickers_download:
        tickers_download.append('BRL=X')

    end_date = datetime.date.today()
    start_date = end_date - datetime.timedelta(days=365*anos_historico + 20)
    start_str_bcb = start_date.strftime('%d/%m/%Y')

    print(f"--- Baixando dados de {len(tickers_download)} ativos + Selic ---")

    # Download Yahoo financa
    try:
        dados_brutos = yf.download(tickers_download, start=start_date, end=end_date, progress=True, auto_adjust=True, threads=True)
    except Exception as e:
        print(f"Erro no Yahoo: {e}")
        return None

    # Tratamento Colunas
    if isinstance(dados_brutos.columns, pd.MultiIndex):
        try:
            dados = dados_brutos['Close'].copy()
        except KeyError:
            dados = dados_brutos['Adj Close'].copy()
    else:
        dados = dados_brutos.copy()

    # B. Download Selic (BCB)
    url_bcb = f'http://api.bcb.gov.br/dados/serie/bcdata.sgs.11/dados?formato=json&dataInicial={start_str_bcb}'
    try:
        selic_df = pd.read_json(url_bcb)
        selic_df['data'] = pd.to_datetime(selic_df['data'], dayfirst=True)
        selic_df = selic_df.set_index('data')
        selic_df['fator'] = 1 + (selic_df['valor'] / 100)
        selic_series = selic_df['fator'].cumprod()
        selic_series.name = 'SELIC_SINTETICA'
    except Exception as e:
        print(f"Erro no BCB: {e}")
        return None

    # C. Integração e Limpeza
    dados.index = dados.index.tz_localize(None)
    dados = pd.concat([dados, selic_series], axis=1)

    # Remove colunas vazias
    dados = dados.dropna(axis=1, how='all')

    # Regra de corte (queremos pelo menos 60% de preenchimento)
    limite_dados = len(dados) * 0.6
    dados = dados.dropna(axis=1, thresh=limite_dados)

    print(f"Ativos válidos após corte de histórico: {len(dados.columns)}")
    dados = dados.ffill()

    # D. Conversão de Moeda (como temos alguns ativos internacionais eles estão em dólar.)
    if tem_internacional:
        if 'BRL=X' not in dados.columns:
            print("AVISO: Dólar não encontrado. Retornando dados originais.")
            return dados.dropna()

        serie_dolar = dados['BRL=X']
        dados = dados.drop(columns=['BRL=X'], errors='ignore')

        cols_br = [c for c in dados.columns if c.endswith('.SA') or c == 'SELIC_SINTETICA']
        cols_us = [c for c in dados.columns if c not in cols_br]

        df_br = dados[cols_br]
        df_us = dados[cols_us]

        df_us_convertido = df_us.mul(serie_dolar, axis=0)
        df_us_convertido.columns = [f"{c}_BRL" for c in df_us_convertido.columns]

        df_final = pd.concat([df_br, df_us_convertido], axis=1)
    else:
        df_final = dados


    return df_final.dropna()

In [ ]:
# @title Baixando os dados dos ativos
meus_tickers = gerar_lista_simples()

# Passo 2: Baixar e processar
data_final = obter_dados_final(meus_tickers, anos_historico=quantidade_de_anos) # Usei 3 anos para garantir mais ativos

if data_final is not None and not data_final.empty:
    # Passo 3: Preparar Matrizes
    returns = data_final.pct_change().dropna()
    # selic_aa_target = 0.15 # aqui eu estou forçando para que a selic seja 15% a.a, independente da média, isso estava influenciando no retorno das carteiras, isso quando incluíamos a selic
    # selic_daily = (1 + selic_aa_target)**(1/252) - 1
    # returns['SELIC_SINTETICA'] = selic_daily
    returns_np = returns.values
    T, N = returns_np.shape
    mu = returns.mean().values
    mu_norm = (mu - mu.mean())/mu.std()
    Sigma = returns.cov().values

    # Passo 4: Atualizar variáveis para o Gurobi
    tickers = data_final.columns.tolist()
    n = len(tickers)

    print("-" * 40)
    print(f"Total de Ativos: {n}")
    print(f"Selic inclusa? {'SELIC_SINTETICA' in tickers}")
    print(f"Exemplo de ativos: {tickers[:5]} ... {tickers[-1]}")
else:
    print("Falha na aquisição de dados.")

In [ ]:
# @title Classificando os ativos em setores
from tqdm import tqdm

# Verifica se data_final existe.
if 'data_final' not in locals() or data_final is None:
    print("ERRO: A variável 'data_final' não existe. Rode o código de download primeiro.")
    # Cria lista vazia para não quebrar o código abaixo se for só teste
    lista_tickers_modelo = []
else:
    lista_tickers_modelo = data_final.columns.tolist()

def enrich_row(ticker_modelo):
    """
    ticker_modelo: O nome da coluna no seu DataFrame (ex: 'AAPL_BRL', 'PETR4.SA')
    """
    # 1. Define valores padrão
    out = {
        'ticker': ticker_modelo,
        'setor_yf': None,
        'pais_yf': 'Brasil' # Assume Brasil por padrão
    }

    # 2. Tratamento do nome para o Yahoo Finance
    # Remove o sufixo _BRL para a busca, mas mantém no output para o Gurobi usar depois
    ticker_busca = ticker_modelo.replace('_BRL', '')

    # 3. Exceções manuais (Selic e Cripto as vezes dão erro na busca de info)
    if 'SELIC' in ticker_busca:
        out['setor_yf'] = 'Renda Fixa / Caixa'
        out['pais_yf'] = 'Brasil'
        return out

    if '-USD' in ticker_busca:
        out['setor_yf'] = 'Criptoativos'
        out['pais_yf'] = 'Global'
        return out

    # 4. Buscando no Yahoo
    try:
        tk = yf.Ticker(ticker_busca)
        # Tenta pegar a info
        info = tk.info
        sector = info.get('sector')
        country = info.get('country')

        # Refinamento para FIIs (O Yahoo as vezes chama FII de "Real Estate" genérico ou "Financial")
        if ticker_busca.endswith('11.SA') and sector in ['Financial Services', 'Real Estate', None, 'Financial']:
            # se tem 11.SA e não é um ETF conhecido, grande chance de ser FII
            sector = 'Fundo Imobiliário'

        out['setor_yf'] = sector
        out['pais_yf'] = country

    except Exception:
        # Se der algum erro de conexão ou ticker não encontrado
        out['setor_yf'] = 'erro'
        pass

    return out

# --- Execução ---

print(f"Iniciando coleta de setores para {len(lista_tickers_modelo)} ativos...")

enriched = []

for t in tqdm(lista_tickers_modelo):
    e = enrich_row(t)
    enriched.append(e)

df_setores = pd.DataFrame(enriched)

traducao_setores = {
    'Financial Services': 'Financeiro',
    'Technology': 'Tecnologia',
    'Basic Materials': 'Materiais Básicos',
    'Energy': 'Energia',
    'Consumer Cyclical': 'Consumo Cíclico',
    'Consumer Defensive': 'Consumo Não-Cíclico',
    'Healthcare': 'Saúde',
    'Industrials': 'Industrial',
    'Real Estate': 'Imobiliário',
    'Utilities': 'Utilidade Pública',
    'Communication Services': 'Comunicações',
    'Fundo Imobiliário': 'Fundo Imobiliário',
    'Criptoativos': 'Criptoativos',
    'Renda Fixa / Caixa': 'Renda Fixa / Caixa',
    '': 'ETF'
}

df_setores['setor_traduzido'] = df_setores['setor_yf'].map(traducao_setores).fillna(df_setores['setor_yf'])
df_setores['setor_traduzido'] = df_setores['setor_traduzido'].fillna('Outros/ETF') # O que sobrar é ETF ou erro


# Salvar
df_setores.to_csv('mapeamento_setores_final.csv', index=False)

print("\n--- Amostra dos Dados ---")
print(df_setores.head())
print("\n--- Contagem por Setor ---")
print(df_setores['setor_traduzido'].value_counts())

In [ ]:
# @title Cálculo dos custos dinâmicos de acordo com o capital inicial
def calcular_custos_dinamicos(capital):
    """
    Define os custos fixos e proporcionais dinamicamente
    baseado no tamanho do capital investido (Regra de Faixas).
    """
    #USAMOS OS CUSTOS BASEADOS NAS TAXAS DA "XP INVESTIMENTOS"
    if capital < 1000.00:
        perfil = "Iniciante"
        custo_fixo = 4.90
        aliquota_proporcional = 0.030

    elif capital < 50000.00:
        perfil = "Intermediário"
        custo_fixo = 2.50
        aliquota_proporcional = 0.025

    else:
        perfil = "Alta Renda / VIP"
        custo_fixo = 0.00
        aliquota_proporcional = 0.020

    # 2. CÁLCULO MATEMÁTICO
    custo_proporcional_reais = capital * (aliquota_proporcional / 100)
    custo_total = custo_fixo + custo_proporcional_reais
    percentual_total = (custo_total / capital) * 100
    return custo_fixo, aliquota_proporcional, (custo_fixo/capital)

CAPITAL_INICIAL = 1000.00
CUSTO_FIXO, CUSTO_PROP, FATOR_CUSTO_FIXO =calcular_custos_dinamicos(CAPITAL_INICIAL)
print(calcular_custos_dinamicos(CAPITAL_INICIAL))

# Omega Ratio (Charnes-Cooper) — com Custos

Código de Otimização de Portfólio Mista-Inteira (MIP) utilizando o solver Gurobi.

O objetivo central é maximizar o Omega Ratio de uma carteira, considerando restrições do mundo real que tornam o problema não-linear ou não-convexo, especificamente: Custos de Transação (Fixos e Variáveis) e Cardinalidade (Número máximo de ativos). Além disso utilizamos da tranformação de Charnes-Cooper.


## Variáveis de Decisão
* $w_i$: Peso original do ativo $i$.
* $y_i = \tau w_i$: Variável de peso transformada (escala $\tau$).
* $\tau > 0$: Variável de escala (inverso do denominador original).
* $\nu_t \ge 0$: Desvio positivo (upside) no cenário $t$.
* $\delta_t \ge 0$: Desvio negativo (downside) no cenário $t$.
* $z_i \in \{0,1\}$: Variável binária de seleção do ativo $i$.

## Parâmetros
* $r_{t,i}$: Retorno histórico do ativo $i$ no cenário $t$.
* $L$: *Threshold* (limiar) de retorno exigido para o cálculo do Omega.
* $c_i$: Custo proporcional de transação.
* $f_i$: Custo fixo incorrido ao selecionar o ativo.
* $K_{\max}$: Cardinalidade máxima da carteira.

## Transformação dos Retornos

O retorno líquido original no cenário $t$ é dado por:

$$
R_t(w,z) = \sum_{i} r_{t,i} w_i \;-\; \sum_{i} c_i w_i \;-\; \sum_{i} f_i z_i
$$

Após a transformação de Charnes-Cooper (multiplicação por $\tau$), o retorno transformado torna-se:

$$
R_t(y,\tau,z) = \sum_{i} (r_{t,i} - c_i) y_i \;-\; \tau \sum_{i} f_i z_i
$$

## Modelo Matemático Transformado

O objetivo é maximizar a soma dos desvios positivos (numerador do Omega), fixando a soma dos desvios negativos em 1 (denominador):

$$
\begin{aligned}
\textbf{Maximizar:} \quad & \sum_{t} \nu_t & \\
\textbf{Sujeito a:} \quad & \sum_{i} (r_{t,i} - c_i) y_i - \tau \sum_{i} f_i z_i - L\tau = \nu_t - \delta_t, \quad \forall t & \text{(1) Definição dos Desvios} \\
& \sum_{t} \delta_t = 1 & \text{(2) Normalização} \\
& \sum_{i} y_i = \tau & \text{(3) Orçamento Transformado} \\
& \sum_{i} z_i \le K_{\max} & \text{(4) Cardinalidade} \\
& l \tau z_i \le y_i \le u \tau z_i, \quad \forall i & \text{(5) Limites Transformados} \\
& \sum_{i \in \text{setor } s} y_i \le L_s \tau, \quad \forall s \in S \quad & \text{(6) Limite Setorial} \\
& y_i \ge 0, \quad \nu_t \ge 0, \quad \delta_t \ge 0, \quad \tau > 0 & \text{(Domínio)}
\end{aligned}
$$

In [ ]:
# @title Omega Ratio
# Parâmetros de Custos
CAPITAL_INICIAL = 100000.00
CUSTO_FIXO, CUSTO_PROP, FATOR_CUSTO_FIXO = calcular_custos_dinamicos(CAPITAL_INICIAL)

# Dados do Omega
TARGET_RETURN_AA = 0.15     # Meta Anual (15%)
THRESHOLD_L = 0.00055131    # L (Limiar diário - ex: selic atual)

# Atualiza dimensões
if 'returns_np' in locals():
    T_real, N_real = returns_np.shape
else:
    raise ValueError("Matriz 'returns_np' não encontrada. Rode o download de dados primeiro.")

# Mapeamento de Setores
df_setores = pd.read_csv('mapeamento_setores_final.csv')
mapa_setores = dict(zip(df_setores['ticker'], df_setores['setor_traduzido']))
setores_unicos = df_setores['setor_traduzido'].unique()
lista_setores_ordenada = [mapa_setores.get(t, 'Outros') for t in tickers]

# dicionário setor → índices dos ativos
indices_por_setor = {}
for setor in setores_unicos:
    indices = [i for i, s in enumerate(lista_setores_ordenada) if s == setor]
    if indices:
        indices_por_setor[setor] = indices

# Configuração de Limites
limites_config = {
    'Financeiro': 0.30,
    'Energia': 0.35,
    'Materiais Básicos': 0.35,
    'Utilidade Pública': 0.15,
    'Tecnologia': 0.35,
    'Renda Fixa / Caixa': 0,
    'Criptoativos': 0.10,
    'Consumo Cíclico': 0.25,
    'Consumo Não-Cíclico': 0.10,
    'Saúde': 0.1,
    'Industrial': 0.25,
    'Imobiliário': 0.15,
    'Comunicações': 0.15,
    'Fundo Imobiliário': 0.20,
    'ETF': 0.20
}

# Parâmetros da Otimização
LIMIT_PER_ASSET = 0.20
MIN_WEIGHT_IF_SELECTED = 0.03
MAX_ASSETS = 13
M_BIG = 1000


def solve_omega_gurobi_custos(sector_indices_dict, sector_limits_dict):
    print("--- Iniciando Omega Ratio ---")

    try:
        m = gp.Model("Omega_Ratio_Custos", env=env)
        m.setParam('OutputFlag', 1)
        m.setParam('TimeLimit', 60)

        # Vetor de médias diárias
        mu_vec = returns_np[:, :N_real].mean(axis=0)

        # --- VARIÁVEIS ---
        y = m.addMVar(shape=N_real, lb=0, name="y")       # y = w * tau
        z = m.addMVar(shape=N_real, vtype=GRB.BINARY, name="z") # Seleção
        tau = m.addVar(lb=0, name="tau")

        # Variáveis do Omega (Upside/Downside)
        nu = m.addMVar(shape=T_real, lb=0, name="nu")
        delta = m.addMVar(shape=T_real, lb=0, name="delta")

        # Variável Auxiliar para Linearização do Custo Fixo (z * tau)
        z_tau = m.addMVar(shape=N_real, lb=0, name="z_tau")

        # --- FUNÇÃO OBJETIVO ---
        m.setObjective(nu.sum(), GRB.MAXIMIZE)

        # --- RESTRIÇÕES DE LINEARIZAÇÃO (z_tau) ---
        # z_tau representa z[i] * tau.
        # Como tau é pequeno (geralmente < 10), M_TAU=100 é seguro.
        M_TAU = 1000000
        for i in range(N_real):
            m.addConstr(z_tau[i] <= M_TAU * z[i])      # Se z=0 -> z_tau=0
            m.addConstr(z_tau[i] <= tau)               # Se z=1 -> z_tau <= tau
            m.addConstr(z_tau[i] >= tau - M_TAU * (1 - z[i])) # Se z=1 -> z_tau >= tau

        # --- RESTRIÇÕES OMEGA PADRÃO ---
        m.addConstr(delta.sum() == 1, name="Risk_Norm")
        m.addConstr(y.sum() == tau, name="Budget") # sum(y) = tau implica sum(w)=1

        # 1. Retorno Anual Bruto Transformado: sum(y * mu * 252)
        ret_anual_bruto_t = gp.quicksum(y[i] * mu_vec[i] * 252 for i in range(N_real))

        # 2. Custos Transformados
        # Custo Prop: incide sobre o volume total (tau)
        custo_prop_t = gp.quicksum(y[i] * CUSTO_PROP for i in range(N_real))
        # Custo Fixo: incide sobre a contagem de ativos (z_tau)
        custo_fixo_t = gp.quicksum(z_tau[i] * FATOR_CUSTO_FIXO for i in range(N_real))

        # 3. A Restrição
        m.addConstr(
            ret_anual_bruto_t - custo_prop_t - custo_fixo_t >= TARGET_RETURN_AA * tau,
            name="Retorno_Min_Liq_Anual"
        )

        # Mantemos a dinâmica diária original para calcular o Omega
        # sum(y * r_t) - L * tau = nu - delta
        for t in range(T_real):
            r_t = returns_np[t, :]
            expr_retorno_dia = gp.quicksum(y[i] * r_t[i] for i in range(N_real))
            m.addConstr(expr_retorno_dia - (THRESHOLD_L * tau) == nu[t] - delta[t])

        # --- CARDINALIDADE E LIMITES ---
        m.addConstr(z.sum() <= MAX_ASSETS, name="Card_Max")

        # (Big-M)
        for i in range(N_real):
            m.addConstr(y[i] <= LIMIT_PER_ASSET * tau)
            m.addConstr(y[i] <= M_BIG * z[i])
            m.addConstr(y[i] >= (MIN_WEIGHT_IF_SELECTED * tau) - (M_BIG * (1 - z[i])))

        # Setores
        for setor, indices in sector_indices_dict.items():
            limite = sector_limits_dict.get(setor, 0.15)
            m.addConstr(y[indices].sum() <= limite * tau)

        m.optimize()

        if m.status == GRB.OPTIMAL:
            tau_val = tau.X
            if tau_val < 1e-9: return 0, None, None

            # Pesos reais
            w_final = y.X / tau_val
            z_final = z.X
            return m.objVal, w_final, z_final
        else:
            print(f"Status Gurobi: {m.status}")
            if m.status == GRB.INFEASIBLE: m.computeIIS(); m.write("model.ilp")
            return 0, None, None

    except Exception as e:
        print(f"Erro: {e}")
        return 0, None, None

omega_val, weights_opt, z_opt = solve_omega_gurobi_custos(indices_por_setor, limites_config)

if weights_opt is not None:
    # Realizando cálculos das métricas
    tickers_atuais = tickers[:N_real]
    num_ativos = int(sum(z_opt))

    # Retorno Bruto
    mu_daily = returns_np[:, :N_real].mean(axis=0)
    ret_dia_bruto = np.dot(weights_opt, mu_daily)
    ret_ano_bruto = ret_dia_bruto * 252 # Linearização simples para visualização

    # Custos
    custo_fixo_reais = num_ativos * CUSTO_FIXO
    custo_prop_reais = CAPITAL_INICIAL * CUSTO_PROP
    custo_total_reais = custo_fixo_reais + custo_prop_reais
    custo_total = custo_total_reais / CAPITAL_INICIAL

    # Retorno Líquido
    ret_ano_liquido = ret_ano_bruto - custo_total

    # Volatilidade
    sigma_matrix = np.cov(returns_np[:, :N_real], rowvar=False)
    vol_dia = np.sqrt(np.dot(weights_opt.T, np.dot(sigma_matrix, weights_opt)))
    vol_ano = vol_dia * np.sqrt(252)

    sharpe = (ret_ano_liquido - TARGET_RETURN_AA) / vol_ano

    # --- DATAFRAME CARTEIRA ---
    carteira_dados = []
    print(f"\n{'='*60}")
    print(f"{' RELATÓRIO OMEGA COM CUSTOS ':^60}")
    print(f"{'='*60}")

    print(f"\n>>> RESUMO FINANCEIRO")
    print(f"Capital Inicial:      R$ {CAPITAL_INICIAL:,.2f}")
    print(f"Ativos Selecionados:  {num_ativos}")
    print(f"Custo Total (Taxas):  R$ {custo_total_reais:,.2f} ({custo_total:.4%})")
    print("-" * 40)
    print(f"Omega Ratio:          {omega_val:.4f}")
    print(f"Retorno BRUTO (a.a):  {ret_ano_bruto:.2%}")
    print(f"Retorno LÍQ. (a.a):   {ret_ano_liquido:.2%}")
    print(f"Volatilidade (a.a):   {vol_ano:.2%}")
    print(f"Sharpe (Líquido):     {sharpe:.2f}")

    print(f"\n>>> DETALHE DA CARTEIRA")
    print(f"{'ATIVO':<10} | {'SETOR':<20} | {'PESO (%)':<10} | {'VALOR (R$)'}")
    print("-" * 60)

    for i in range(N_real):
        if z_opt[i] > 0.5:
            peso = weights_opt[i]
            val = peso * CAPITAL_INICIAL
            setor = lista_setores_ordenada[i]
            print(f"{tickers[i]:<10} | {setor:<20} | {peso:7.2%} | R$ {val:,.2f}")
            carteira_dados.append({'Ativo': tickers[i], 'Setor': setor, 'Peso': peso})

    # --- GRÁFICOS ---
    if carteira_dados:
        df_plot = pd.DataFrame(carteira_dados)
        fig, ax = plt.subplots(1, 2, figsize=(16, 7))

        # Donut
        ax[0].pie(df_plot['Peso'], labels=df_plot['Ativo'], autopct='%1.1f%%',
                  startangle=90, pctdistance=0.85, explode=[0.05]*len(df_plot))
        ax[0].add_artist(plt.Circle((0,0),0.70,fc='white'))
        ax[0].set_title('Alocação por Ativos (Omega)')

        # Barras Setor
        df_setor = df_plot.groupby('Setor')['Peso'].sum().sort_values()
        bars = ax[1].barh(df_setor.index, df_setor.values, color=plt.cm.Paired(np.linspace(0,1,len(df_setor))))
        ax[1].bar_label(bars, fmt='%.2f%%', padding=3)
        ax[1].set_title('Exposição Setorial')
        ax[1].set_xlim(0, df_setor.max()*1.2)

        plt.tight_layout()
        plt.show()
else:
    print("Inviável ou erro na otimização.")

# CVaR (Rockafellar–Uryasev) — com Custos

## Variáveis de Decisão
* $w_i \ge 0$: Peso investido no ativo $i$.
* $z_i \in \{0,1\}$: Variável binária de seleção do ativo $i$.
* $\eta$: Variável auxiliar (representa o VaR aproximado no ótimo).
* $p_t \ge 0$: Variável auxiliar que captura o excesso de perda no cenário $t$ (tail loss).

## Parâmetros
* $r_{t,i}$: Retorno histórico do ativo $i$ no cenário $t$.
* $c_i$: Custo proporcional de transação.
* $f_i$: Custo fixo incorrido ao selecionar o ativo.
* $\alpha$: Nível de confiança do CVaR (ex: $0.95$ ou $0.99$).
* $T$: Número total de cenários históricos.
* $S$: Conjunto de setores econômicos.
* $L_s$: Limite de exposição ao setor $s$.

## Definição de Perda
Primeiro, definimos o **Retorno Líquido** ($R_t$) e a **Perda Líquida** ($L_t$) para cada cenário $t$. A perda é o negativo do retorno líquido:

$$
R_t(w,z) = \sum_{i} r_{t,i} w_i \;-\; \sum_{i} c_i w_i \;-\; \sum_{i} f_i z_i
$$

$$
L_t(w,z) = - R_t(w,z) = \sum_{i} (c_i - r_{t,i}) w_i \;+\; \sum_{i} f_i z_i
$$

## Modelo Matemático

A Função Objetivo minimiza o CVaR, composto pelo termo do VaR ($\eta$) e a média das perdas de cauda:

$$
\begin{aligned}
\textbf{Minimizar:} \quad & \eta + \frac{1}{(1-\alpha)T} \sum_{t=1}^{T} p_t & \\
\textbf{Sujeito a:} \quad & p_t \ge L_t(w,z) - \eta, \quad \forall t & \text{(1) Restrição do CVaR} \\
& \sum_{i} w_i = 1 & \text{(2) Orçamento} \\
& l z_i \le w_i \le u z_i, \quad \forall i & \text{(3) Limites Semicontínuos} \\
& K_{\min} \le \sum_{i} z_i \le K_{\max} & \text{(4) Cardinalidade} \\
& \sum_{i \in \text{setor } s} w_i \le L_s, \quad \forall s \in S \quad & \text{(5) Limite Setorial} \\
& \sum_{i} \mu_i w_i - \left( \sum_{i} c_i w_i + \sum_{i} f_i z_i \right) \ge R_{min} & \text{(6) Retorno Mínimo Líquido} \\
& w_i \ge 0, \quad p_t \ge 0, \quad z_i \in \{0,1\} & \text{(Domínio)}
\end{aligned}
$$

In [ ]:
# @title CVaR

# Meta anual MÍNIMA
TARGET_RETURN_AA = 0.175
# Conversão para meta diária (Juros Compostos -> 252 dias úteis)
TARGET_RETURN_DAILY = (1 + TARGET_RETURN_AA)**(1/252) - 1
print(f"Meta de Retorno Anual: {TARGET_RETURN_AA:.2%}")
print(f"Meta de Retorno Diário: {TARGET_RETURN_DAILY:.6%}")

mu_vec = returns.mean().values

R_mat = returns.values       # matriz (T,n)
T = R_mat.shape[0]
alpha = 0.95                # nível de confiança para o CVaR, pro conservador vamos jogar na faixa 0.975 – 0.990
k_max = 10                   # máximo de ativos permitidos
k_min = 10                   # mínimo de ativos
lower_pct = 0.04             # mínimo por ativo selecionado
upper_pct = 0.6              # máximo por ativo selecionado

l_bounds = np.full(n, lower_pct)
u_bounds = np.full(n, upper_pct)

# Big-M adequado
M = upper_pct                # big-M simples para vinculação


# Construção dos índices por setor
df_setores = pd.read_csv('mapeamento_setores_final.csv')
mapa_setores = dict(zip(df_setores['ticker'], df_setores['setor_traduzido']))
setores_unicos = df_setores['setor_traduzido'].unique()

# lista ordenada dos setores com a ordem dos ticker
lista_setores_ordenada = [mapa_setores.get(t, 'Outros') for t in tickers]

# dicionário setor → índices dos ativos
indices_por_setor = {}
for setor in setores_unicos:
    idx = [i for i, s in enumerate(lista_setores_ordenada) if s == setor]
    if idx:
        indices_por_setor[setor] = idx

# limites máximos de cada setor
limites_config = {
    'Financeiro': 0.60, #0.1
    'Energia': 0.10,
    'Materiais Básicos': 0.05,
    'Utilidade Pública': 0.15,
    'Tecnologia': 0.05,
    'Renda Fixa / Caixa': 0, #tiramos a selic
    'Criptoativos': 0.00,
    'Consumo Cíclico': 0.05,
    'Consumo Não-Cíclico': 0.10,
    'Saúde': 0.10,
    'Industrial': 0.05,
    'Imobiliário': 0.10,
    'Comunicações': 0.10,
    'Fundo Imobiliário': 0.20,
    'ETF': 0.20
}

# Criar modelo
m = gp.Model("CVaR_Cardinality", env=env)

# Variáveis
w = m.addVars(n, lb=0.0, ub=1.0, name="w")      # pesos
y = m.addVars(n, vtype=GRB.BINARY, name="y")    # ativo selecionado
eta = m.addVar(lb=-GRB.INFINITY, name="eta")    # VaR
z = m.addVars(T, lb=0.0, name="z")              # excessos

# Objetivo: Minimizar CVaR
m.setObjective(
    eta + (1/((1-alpha)*T)) * gp.quicksum(z[t] for t in range(T)),
    GRB.MINIMIZE
)

# Restrições CVaR
for t in range(T):
    m.addConstr(
        z[t] >= -gp.quicksum(w[i] * R_mat[t, i] for i in range(n)) - eta,
        name=f"cvar_{t}"
    )

# Orçamento
m.addConstr(gp.quicksum(w[i] for i in range(n)) == 1, name="budget")

# Cardinalidade
m.addConstr(gp.quicksum(y[i] for i in range(n)) <= k_max,
            name="cardinality_max")

m.addConstr(gp.quicksum(y[i] for i in range(n)) >= k_min,
            name="cardinality_min")

retorno_anual_bruto = gp.quicksum(w[i] * mu_vec[i] * 252 for i in range(n))

# Calcular Custo Total (Pago uma única vez na entrada)
custo_total_entrada = gp.quicksum(w[i] * CUSTO_PROP for i in range(n)) + \
                      gp.quicksum(y[i] * FATOR_CUSTO_FIXO for i in range(n))

# Restrição: O ganho de 1 ano menos os custos deve bater a meta anual
m.addConstr(
    retorno_anual_bruto - custo_total_entrada >= TARGET_RETURN_AA,
    name="Retorno_Min_Liq_Anualizado"
)

#limites condicionados à seleção
for i in range(n):
    m.addConstr(w[i] <= u_bounds[i] * y[i], name=f"upper_link_{i}")
    m.addConstr(w[i] >= l_bounds[i] * y[i], name=f"lower_link_{i}")

# LIMITES DE SETOR
for setor, indices in indices_por_setor.items():
    if setor in limites_config:
        m.addConstr(
            gp.quicksum(w[i] for i in indices) <= limites_config[setor],
            name=f"setor_max_{setor}"
        )

# Parâmetros do solver
m.Params.TimeLimit = 300
m.Params.MIPGap = 1e-4
m.Params.OutputFlag = 1

m.optimize()

if m.status in [GRB.OPTIMAL, GRB.SUBOPTIMAL]:
        print("\n" + "="*60)
        print(f"{' RELATÓRIO CVaR COM CUSTOS ':^60}")
        print("="*60)

        # Recuperar valores
        w_opt = np.array([w[i].X for i in range(n)])
        y_opt = np.array([y[i].X for i in range(n)]) # 0 ou 1
        cvar_val = m.ObjVal

        #CÁLCULOS
        num_ativos = int(sum(y_opt))

        # Retorno Bruto (Baseado apenas nos pesos)
        port_ret_diario_bruto = np.dot(mu_vec, w_opt)
        port_ret_anual_bruto = (1 + port_ret_diario_bruto)**252 - 1

        # Custos Totais
        custo_fixo_total_pct = num_ativos * FATOR_CUSTO_FIXO
        custo_prop_total_pct = w_opt.sum() * CUSTO_PROP # w_opt.sum() ~ 1.0
        custo_total_pct = custo_fixo_total_pct + custo_prop_total_pct

        # Retorno Líquido
        # Nota: Subtraímos o custo total do retorno anualizado bruto aproximado
        port_ret_anual_liquido = port_ret_anual_bruto - custo_total_pct

        # Volatilidade
        var_dia = np.dot(w_opt.T, np.dot(returns.cov().values, w_opt))
        vol_aa = np.sqrt(var_dia) * np.sqrt(252)

        # --- 2. PRINT DA INTERFACE ---
        print(f"\n>>> RESUMO FINANCEIRO")
        print(f"Capital Inicial:      R$ {CAPITAL_INICIAL:,.2f}")
        print(f"Número de Ativos:     {num_ativos}")
        print(f"Custo Total (Taxas):  R$ {(custo_total_pct * CAPITAL_INICIAL):,.2f} ({custo_total_pct:.4%})")
        print("-" * 40)
        print(f"CVaR ({alpha*100:.1f}%):       {cvar_val:.4f} (Diário)")
        print(f"Retorno BRUTO Esp.:   {port_ret_anual_bruto:.2%} (a.a.)")
        print(f"Retorno LÍQUIDO Esp.: {port_ret_anual_liquido:.2%} (a.a.)")
        print(f"Volatilidade:         {vol_aa:.2%} (a.a.)")

        sharpe = (port_ret_anual_liquido - TARGET_RETURN_AA)/vol_aa if vol_aa > 0 else 0
        print(f"Sharpe (Líquido):     {sharpe:.2f}")

        # --- 3. DETALHE DA CARTEIRA ---
        carteira_dados = []
        print(f"\n>>> DETALHE DA CARTEIRA")
        print(f"{'ATIVO':<10} | {'SETOR':<20} | {'PESO (%)':<10} | {'VALOR (R$)'}")
        print("-" * 60)

        for i in range(n):
            if y_opt[i] > 0.5:
                peso = w_opt[i]
                valor_alocado = peso * CAPITAL_INICIAL
                setor = lista_setores_ordenada[i]
                print(f"{tickers[i]:<10} | {setor:<20} | {peso:7.2%} | R$ {valor_alocado:,.2f}")

                carteira_dados.append({
                    'Ativo': tickers[i],
                    'Setor': setor,
                    'Peso': peso,
                    'Valor': valor_alocado
                })

        # --- 4. GRÁFICOS ---
        if carteira_dados:
            df_plot = pd.DataFrame(carteira_dados)

            fig, ax = plt.subplots(1, 2, figsize=(16, 7))

            # Gráfico 1: Donut de Ativos
            wedges, texts, autotexts = ax[0].pie(
                df_plot['Peso'],
                labels=df_plot['Ativo'],
                autopct='%1.1f%%',
                startangle=90,
                pctdistance=0.85,
                explode=[0.05]*len(df_plot)
            )
            ax[0].add_artist(plt.Circle((0,0),0.70,fc='white'))
            ax[0].set_title(f'Alocação CVaR Otimizada\n(Capital: R$ {CAPITAL_INICIAL:,.0f})', fontsize=14)

            # Gráfico 2: Barras de Setores
            df_setor = df_plot.groupby('Setor')['Peso'].sum().sort_values()
            cores_setor = plt.cm.Paired(np.linspace(0, 1, len(df_setor)))

            bars = ax[1].barh(df_setor.index, df_setor.values, color=cores_setor)
            ax[1].set_title('Exposição Setorial (CVaR)', fontsize=14)
            ax[1].set_xlabel('Peso (%)')
            ax[1].bar_label(bars, fmt='%.2f%%', padding=3)
            ax[1].set_xlim(0, df_setor.max() * 1.2)

            plt.tight_layout()
            plt.show()
        else:
            print("Carteira vazia (verifique se o modelo convergiu).")

else:
      print(f"Solução não encontrada. Status: {m.status}")



# Markowitz com Cardinalidade (MIQP) — com Custos

## Variáveis de Decisão
* $w_i \ge 0$: Peso (proporção) investido no ativo $i$.
* $z_i \in \{0,1\}$: Variável binária que indica se o ativo $i$ é selecionado ($1$ se sim, $0$ caso contrário).

## Parâmetros
* $\Sigma$: Matriz de covariância dos retornos (anualizada).
* $\mu_i$: Retorno esperado anualizado do ativo $i$.
* $c_i$: Custo proporcional de transação (por unidade de peso investido) no ativo $i$.
* $f_i$: Custo fixo incorrido ao selecionar o ativo $i$.
* $l, u$: Limites inferior e superior de alocação por ativo (se selecionado).
* $K_{\min}, K_{\max}$: Cardinalidade mínima e máxima (número de ativos na carteira).
* $\lambda$: Parâmetro de aversão ao risco do investidor.
* $S$: Conjunto de setores econômicos.
* $L_s$: Limite máximo de alocação para o setor $s$.

## Definições Auxiliares
O **Retorno Líquido** ($R_{\text{liq}}$), descontando os custos de transação e fixos, é definido como:

$$
R_{\text{liq}}(w,z) = \sum_{i} \mu_i w_i \;-\; \sum_{i} c_i w_i \;-\; \sum_{i} f_i z_i
$$

## Modelo Matemático

A Função Objetivo visa minimizar a variância da carteira (risco) subtraída do retorno líquido ponderado pela aversão ao risco:

$$
\begin{aligned}
\textbf{Minimizar:} \quad & w^\top \Sigma w \;-\; \lambda \, R_{\text{liq}}(w,z) & \\
\textbf{Sujeito a:} \quad & \sum_{i} w_i = 1 & \text{(1) Orçamento} \\
& l z_i \le w_i \le u z_i, \quad \forall i & \text{(2) Limites Semicontínuos} \\
& K_{\min} \le \sum_{i} z_i \le K_{\max} & \text{(3) Cardinalidade} \\
& \sum_{i \in \text{setor } s} w_i \le L_s, \quad \forall s \in S \quad & \text{(4) Limite Setorial} \\
& w_i \ge 0, \quad z_i \in \{0,1\}, \quad \forall i & \text{(Domínio das variáveis)}
\end{aligned}
$$

In [ ]:
# @title Markowitz


df_setores = pd.read_csv('mapeamento_setores_final.csv')
mapa_setores = dict(zip(df_setores['ticker'], df_setores['setor_traduzido']))
setores_unicos = df_setores['setor_traduzido'].unique()
lista_setores_ordenada = [mapa_setores.get(t, 'Outros') for t in tickers]

limites_config = {
    'Financeiro': 0.2,
    'Energia': 0.18,
    'Materiais Básicos': 0.12,
    'Utilidade Pública': 0.15,
    'Tecnologia': 0.08,
    'Renda Fixa / Caixa': 0, #ESTAMOS TIRANDO A SELIC
    'Criptoativos': 0.03,
    'Consumo Cíclico': 0.08,
    'Consumo Não-Cíclico': 0.12,
    'Saúde': 0.1,
    'Industrial': 0.1,
    'Imobiliário': 0.08,
    'Comunicações': 0.05,
    'Fundo Imobiliário': 0.25,
    'ETF': 0.15
}

TAXA_LIVRE_RISCO_AA = 0.15  # (Selic aproximada)
print(f"Meta de Retorno Mínimo (a.a.): {TAXA_LIVRE_RISCO_AA:.2%}")
mu = returns.mean().values
mu_anualizada = mu * 252  # Assumindo 252 dias por ano
Sigma_anualizada = Sigma * 252  # Assumindo 252 dias por ano
lambda_risk = 0.2

try:
    m = gp.Model("markowitz-custos", env=env)

    # Variáveis
    x = m.addMVar(shape=n, lb=0, ub=1, name="x")
    z = m.addMVar(shape=n, vtype=GRB.BINARY, name="z")

    # Custo Proporcional: Incide sobre o total alocado (sum(x)).
    penalidade_prop = CUSTO_PROP * x.sum()

    # Custo Fixo: Incide sobre cada ativo escolhido (z=1)
    penalidade_fixa = FATOR_CUSTO_FIXO * z.sum()

    # Retorno Líquido Esperado (Anualizado)
    #assumimos que o custo é pago na entrada e afeta a rentabilidade do período
    retorno_bruto = mu_anualizada @ x
    retorno_liquido = retorno_bruto - penalidade_prop - penalidade_fixa

    # --- FUNÇÃO OBJETIVO ---
    # Min Risco - lambda * Retorno Líquido
    # (Ou Max Retorno Líquido - lambda * Risco)
    risco = x @ Sigma_anualizada @ x
    m.setObjective(risco - (lambda_risk * retorno_liquido), GRB.MINIMIZE)

    # RESTRIÇÕES
    m.addConstr(x.sum() == 1, name="Orcamento")

    #Restrições Setoriais
    for s in setores_unicos:
            # Pega os índicesa onde o setor é igual a 's'
            indices = [i for i, setor_ativo in enumerate(lista_setores_ordenada) if setor_ativo == s]
            if len(indices) > 0:
                # Usa o .get() no dicionário correto 'limites_config'
                limite = limites_config.get(s, 1.0)
                m.addConstr(x[indices].sum() <= limite, name=f"lim_setor_{s}")

    # Cardinalidade e Ligação
    u = 0.5 #upper
    l = 0.05 #lower
    K_min = 10
    K_max = 10

    m.addConstr(z.sum() >= K_min, name="Card_min")
    m.addConstr(z.sum() <= K_max, name="Card_max")

    for i in range(n):
        m.addConstr(x[i] <= u * z[i], name=f"upper_bound_{i}")
        m.addConstr(x[i] >= l * z[i], name=f"lower_bound_{i}")

    # Restrição de Retorno Mínimo (Líquido)
    m.addConstr(retorno_liquido >= TAXA_LIVRE_RISCO_AA, name="Retorno_Min_Liq")

    m.optimize()

    if m.status == GRB.OPTIMAL:
            print("\n" + "="*60)
            print(f"{' RELATÓRIO DE ALOCAÇÃO COM CUSTOS ':^60}")
            print("="*60)

            # Recupera solução
            x_sol = x.X
            z_sol = z.X

            num_ativos_selecionados = sum([1 for i in z_sol if i > 0.5])

            # Retorno Bruto (Apenas pesos * retorno ativos)
            ret_esperado_dia = mu @ x_sol
            ret_esperado_ano_bruto = ret_esperado_dia * 252

            # Cálculo dos Custos Reais na Solução Otimizada
            custo_fixo_total_pct = num_ativos_selecionados * FATOR_CUSTO_FIXO
            custo_prop_total_pct = x_sol.sum() * CUSTO_PROP # x.sum() deve ser 1

            custo_total_pct = custo_fixo_total_pct + custo_prop_total_pct

            # Retorno Líquido (Bruto - Custos)
            ret_esperado_ano_liquido = ret_esperado_ano_bruto - custo_total_pct

            # Risco
            variancia = x_sol @ Sigma_anualizada @ x_sol
            volatilidade_ano = np.sqrt(variancia)

            # 2. INTERFACE FINANCEIRA (Valores Monetários)
            # --------------------------------------------------------
            print(f"\n>>> RESUMO FINANCEIRO")
            print(f"Capital Inicial:      R$ {CAPITAL_INICIAL:,.2f}")
            print(f"Número de Ativos:     {num_ativos_selecionados}")
            print(f"Custo Total (Taxas):  R$ {(custo_total_pct * CAPITAL_INICIAL):,.2f} ({custo_total_pct:.4%})")
            print("-" * 40)
            print(f"Retorno BRUTO Esp.:   {ret_esperado_ano_bruto:.2%} (a.a.)")
            print(f"Retorno LÍQUIDO Esp.: {ret_esperado_ano_liquido:.2%} (a.a.)")
            print(f"Volatilidade:         {volatilidade_ano:.2%} (a.a.)")
            print(f"Sharpe (Líquido):     {(ret_esperado_ano_liquido - TAXA_LIVRE_RISCO_AA)/volatilidade_ano:.2f}")

            # 3. TABELA DE ALOCAÇÃO
            # --------------------------------------------------------
            carteira_dados = []
            print(f"\n>>> DETALHE DA CARTEIRA")
            print(f"{'ATIVO':<10} | {'SETOR':<20} | {'PESO (%)':<10} | {'VALOR (R$)'}")
            print("-" * 60)

            for i in range(n):
                if z_sol[i] > 0.5:
                    peso = x_sol[i]
                    valor_alocado = peso * CAPITAL_INICIAL
                    setor = lista_setores_ordenada[i]
                    print(f"{tickers[i]:<10} | {setor:<20} | {peso:7.2%} | R$ {valor_alocado:,.2f}")

                    carteira_dados.append({
                        'Ativo': tickers[i],
                        'Setor': setor,
                        'Peso': peso,
                        'Valor': valor_alocado
                    })

            # 4. GRÁFICOS (INTERFACE VISUAL)
            # --------------------------------------------------------
            df_plot = pd.DataFrame(carteira_dados)

            fig, ax = plt.subplots(1, 2, figsize=(16, 7))

            # Gráfico 1: Distribuição por Ativos (Donut Chart)
            wedges, texts, autotexts = ax[0].pie(
                df_plot['Peso'],
                labels=df_plot['Ativo'],
                autopct='%1.1f%%',
                startangle=90,
                pctdistance=0.85,
                explode=[0.05]*len(df_plot) # Separa levemente as fatias
            )
            # Círculo branco no meio para virar Donut
            centre_circle = plt.Circle((0,0),0.70,fc='white')
            ax[0].add_artist(centre_circle)
            ax[0].set_title(f'Alocação por Ativos\n(Capital: R$ {CAPITAL_INICIAL:,.0f})', fontsize=14)

            # Gráfico 2: Distribuição por Setor (Bar Chart)
            df_setor = df_plot.groupby('Setor')['Peso'].sum().sort_values()
            cores_setor = plt.cm.Paired(np.linspace(0, 1, len(df_setor)))

            bars = ax[1].barh(df_setor.index, df_setor.values, color=cores_setor)
            ax[1].set_title('Exposição Setorial', fontsize=14)
            ax[1].set_xlabel('Peso (%)')
            ax[1].bar_label(bars, fmt='%.2f%%', padding=3)
            ax[1].set_xlim(0, df_setor.max() * 1.2) # Margem para o rótulo

            plt.tight_layout()
            plt.show()
    else:
          print("Solução não encontrada ou inviável.")

except Exception as e:
  print(f"Erro na execução: {e}")



#Estudos de casos





In [ ]:
# @title Teste/simulação das carteiras dos modelos em 2024.

# 1. CONFIGURAÇÕES
DATA_CORTE = '2024-05-22'
#DATA_FINAL = '2022-05-22'
# CAPITAL = 100000.00
META_ALVO_AA = 0.15  # 15% a.a.
# CUSTO_FIXO, CUSTO_PROP, FATOR_CUSTO_FIXO =calcular_custos_dinamicos(CAPITAL_INICIAL)

print(f"--- CONFIGURAÇÃO DO BACKTEST ---")
print(f"Data de Corte (Treino/Teste): {DATA_CORTE}")
print(f"Capital Inicial: R$ {CAPITAL:,.2f}")
print(f"Meta de Retorno: {META_ALVO_AA:.1%}")

# 2. PREPARAÇÃO DOS DADOS (SPLIT)
returns.index = pd.to_datetime(returns.index)

# Dados de Treino
ret_train = returns.loc[:DATA_CORTE]
mu_train = ret_train.mean().values
sigma_train = ret_train.cov().values
returns_np_train = ret_train.values # Para CVaR e Omega


# Dados de Teste
ret_test = returns.loc[DATA_CORTE:]
if ret_train.empty or ret_test.empty:
    raise ValueError("Erro: Data de corte inválida ou fora do intervalo dos dados.")
print(f"Dias de Treino: {len(ret_train)} | Dias de Teste: {len(ret_test)}")


# 3. FUNÇÕES DOS SOLVERS (ISSO SÓ SERVE PRO TREINO PORQUE SELECIONAMOS DATAS ESPECÍFICAS)
def run_markowitz_backtest():
    limites_marko = {
      'Financeiro': 0.2,
      'Energia': 0.18,
      'Materiais Básicos': 0.15,
      'Utilidade Pública': 0.15,
      'Tecnologia': 0.1,
      'Renda Fixa / Caixa': 0,
      'Criptoativos': 0.03,
      'Consumo Cíclico': 0.08,
      'Consumo Não-Cíclico': 0.12,
      'Saúde': 0.1,
      'Industrial': 0.1,
      'Imobiliário': 0.08,
      'Comunicações': 0.08,
      'Fundo Imobiliário': 0.15,
      'ETF': 0.12
    }
    # Usa mu_train e sigma_train
    try:
        m = gp.Model("Markowitz_BT", env=env)
        m.Params.OutputFlag = 0
        x = m.addMVar(n, lb=0, ub=1, name="x")
        z = m.addMVar(n, vtype=GRB.BINARY, name="z")

        # Custos
        custos = (CUSTO_PROP * x.sum()) + (FATOR_CUSTO_FIXO * z.sum())

        # Retorno
        ret_ano = (mu_train * 252) @ x
        m.addConstr(ret_ano - custos >= META_ALVO_AA)

        # Risco (Minimizar) lambda = 0.1 para o moderado
        lambda_risk = 0.2
        sigma_train_anu = sigma_train * 252
        mu_train_anu = mu_train * 252

        m.setObjective((x @ (sigma_train_anu) @ x) - (lambda_risk * (mu_train_anu @ x)), GRB.MINIMIZE)
        # Restrições Padrão
        m.addConstr(x.sum() == 1)
        m.addConstr(z.sum() <= 25)
        m.addConstr(z.sum() >= 10)
        for i in range(n):
            m.addConstr(x[i] <= 0.50 * z[i])
            m.addConstr(x[i] >= 0.05 * z[i])

        # Setores
        for setor, indices in indices_por_setor.items():
            if setor in limites_marko:
                m.addConstr(x[indices].sum() <= limites_marko[setor])

        m.optimize()
        return x.X if m.status == GRB.OPTIMAL else None
    except: return None

def run_cvar_backtest():
    # Usa returns_np_train

    limites_cvar = {
        'Financeiro': 0.10,
        'Energia': 0.10,
        'Materiais Básicos': 0.05,
        'Utilidade Pública': 0.15,
        'Tecnologia': 0.05,
        'Renda Fixa / Caixa': 0,
        'Criptoativos': 0.00,
        'Consumo Cíclico': 0.05,
        'Consumo Não-Cíclico': 0.10,
        'Saúde': 0.10,
        'Industrial': 0.05,
        'Imobiliário': 0.10,
        'Comunicações': 0.10,
        'Fundo Imobiliário': 0.20,
        'ETF': 0.20
    }

    try:
        T_t, N_t = returns_np_train.shape
        m = gp.Model("CVaR_BT", env=env)
        m.Params.OutputFlag = 0

        w = m.addVars(N_t, lb=0, ub=1)
        y = m.addVars(N_t, vtype=GRB.BINARY)
        eta = m.addVar(lb=-GRB.INFINITY)
        z_var = m.addVars(T_t, lb=0)

        # Obj CVaR
        alpha = 0.975
        m.setObjective(eta + (1/((1-alpha)*T_t)) * gp.quicksum(z_var[t] for t in range(T_t)), GRB.MINIMIZE)

        for t in range(T_t):
            r_port_t = gp.quicksum(w[i] * returns_np_train[t, i] for i in range(N_t))
            m.addConstr(z_var[t] >= -r_port_t - eta)

        m.addConstr(gp.quicksum(w[i] for i in range(N_t)) == 1)

        # Retorno
        ret_bruto = gp.quicksum(w[i] * mu_train[i] * 252 for i in range(N_t))
        custos = gp.quicksum(w[i] * CUSTO_PROP for i in range(N_t)) + gp.quicksum(y[i] * FATOR_CUSTO_FIXO for i in range(N_t))
        m.addConstr(ret_bruto - custos >= META_ALVO_AA)

        # Card/Setor
        m.addConstr(gp.quicksum(y[i] for i in range(N_t)) == 10)
        # m.addConstr(gp.quicksum(y[i] for i in range(N_t)) >= 10)
        for i in range(N_t):
            m.addConstr(w[i] <= 0.40 * y[i])
            m.addConstr(w[i] >= 0.04 * y[i])

        for setor, indices in indices_por_setor.items():
            if setor in limites_cvar:
                m.addConstr(gp.quicksum(w[i] for i in indices) <= limites_cvar[setor])

        m.optimize()
        return np.array([w[i].X for i in range(N_t)]) if m.status == GRB.OPTIMAL else None
    except: return None

def run_omega_backtest():
    # Usa returns_np_train e Charnes-Cooper

    limites_omega = {
        'Financeiro': 0.30,
        'Energia': 0.35,
        'Materiais Básicos': 0.35,
        'Utilidade Pública': 0.15,
        'Tecnologia': 0.35,
        'Renda Fixa / Caixa': 0,
        'Criptoativos': 0.10,
        'Consumo Cíclico': 0.25,
        'Consumo Não-Cíclico': 0.10,
        'Saúde': 0.1,
        'Industrial': 0.25,
        'Imobiliário': 0.15,
        'Comunicações': 0.15,
        'Fundo Imobiliário': 0.20,
        'ETF': 0.20
    }

    try:
        T_t, N_t = returns_np_train.shape
        m = gp.Model("Omega_BT", env=env)
        m.Params.OutputFlag = 0

        y = m.addMVar(N_t, lb=0) # y = w * tau
        z = m.addMVar(N_t, vtype=GRB.BINARY)
        tau = m.addVar(lb=0)
        nu = m.addMVar(T_t, lb=0)
        delta = m.addMVar(T_t, lb=0)
        z_tau = m.addMVar(N_t, lb=0) # Linearização z*tau

        SELIC_2020 = 0.1375 #média da selic no recorte
        L_diario = (1 + SELIC_2020)**(1/252) - 1
        #L_diario = 0.00055131 # Threshold (aprox CDI diário)

        m.setObjective(nu.sum(), GRB.MAXIMIZE)

        # Linearização z*tau
        M_val = 100
        for i in range(N_t):
            m.addConstr(z_tau[i] <= M_val * z[i])
            m.addConstr(z_tau[i] <= tau)
            m.addConstr(z_tau[i] >= tau - M_val*(1-z[i]))

        # Omega Core
        m.addConstr(delta.sum() == 1)
        m.addConstr(y.sum() == tau)

        # Retorno com custos
        ret_ano_t = gp.quicksum(y[i] * (mu_train[i] * 252) for i in range(N_t))
        custos_t = gp.quicksum(y[i] * CUSTO_PROP for i in range(N_t)) + gp.quicksum(z_tau[i] * FATOR_CUSTO_FIXO for i in range(N_t))
        m.addConstr(ret_ano_t - custos_t >= META_ALVO_AA * tau)

        # Upside/Downside
        for t in range(T_t):
            # y @ r_t
            val_t = gp.quicksum(y[i] * returns_np_train[t, i] for i in range(N_t))
            m.addConstr(val_t - L_diario * tau == nu[t] - delta[t])

        # Card/Setor
        m.addConstr(z.sum() == 10)
        # m.addConstr(z.sum() >= 10)
        for i in range(N_t):
            m.addConstr(y[i] <= 0.20 * z_tau[i]) # usando z_tau para consistência
            m.addConstr(y[i] >= 0.03 * z_tau[i])
        for setor, indices in indices_por_setor.items():
            if setor in limites_omega:
                m.addConstr(y[indices].sum() <= limites_omega[setor] * tau)

        m.optimize()
        if m.status == GRB.OPTIMAL and tau.X > 1e-6:
            return y.X / tau.X
        return None
    except: return None

# ==========================================
# 4. EXECUÇÃO DOS MODELOS
# ==========================================
resultados = {}

print("\nRodando Markowitz...")
w_mk = run_markowitz_backtest()
if w_mk is not None: resultados['Markowitz'] = w_mk

print("Rodando CVaR...")
w_cv = run_cvar_backtest()
if w_cv is not None: resultados['CVaR'] = w_cv

print("Rodando Omega Ratio...")
w_om = run_omega_backtest()
if w_om is not None: resultados['Omega'] = w_om

# Adicionar Benchmark (Equally Weighted)
w_ew = np.ones(n) / n
resultados['Benchmark (1/N)'] = w_ew

# ==========================================
# 5. SIMULAÇÃO E PLOTAGEM
# ==========================================
if len(resultados) > 1: # Garante que rodou pelo menos o benchmark e um modelo
    plt.figure(figsize=(14, 7))

    df_performance = pd.DataFrame(index=ret_test.index)
    resumo_final = []

    TRADING_DAYS = 252
    anos_no_teste = len(ret_test) / TRADING_DAYS

    for nome, pesos in resultados.items():
        # Calcula retorno diário da carteira no período de teste
        # Se o peso tiver tamanho diferente (por limpeza de dados), ajusta
        if len(pesos) == ret_test.shape[1]:
            daily_ret = ret_test.dot(pesos)

            # Curva acumulada
            cumulative = (1 + daily_ret).cumprod() * 100
            df_performance[nome] = cumulative



            # Plot
            plt.plot(cumulative, label=nome, linewidth=2 if nome != 'Benchmark (1/N)' else 1.5,
                     linestyle='--' if nome == 'Benchmark (1/N)' else '-')

            # Métricas Finais
            ret_total_abs = (cumulative.iloc[-1] / 100) - 1
            vol_anual = daily_ret.std() * np.sqrt(TRADING_DAYS)
            if anos_no_teste > 0:
                cagr = (1 + ret_total_abs) ** (1 / anos_no_teste) - 1
            else:
                cagr = 0
            sharpe = (cagr - META_ALVO_AA) / vol_anual if vol_anual > 0 else 0



            resumo_final.append({
                'Modelo': nome,
                'Retorno Total (Acum)': ret_total_abs,
                'Retorno Anual (CAGR)': cagr,
                'Volatilidade (a.a.)': vol_anual,
                'Sharpe (a.a.)': sharpe
            })
        else:
            print(f"Aviso: Dimensão de pesos incorreta para {nome}")

    plt.title(f'Backtest: {DATA_CORTE} até Hoje (Anualizado)', fontsize=16)
    plt.ylabel('Evolução do Capital (Base 100)')
    plt.legend()
    plt.grid(True, alpha=0.3)

    print("\n=== RESULTADO FINAL (ANUALIZADO) ===")
    df_resumo = pd.DataFrame(resumo_final).set_index('Modelo')

    # Ordenar pelo Sharpe
    df_resumo = df_resumo.sort_values('Sharpe (a.a.)', ascending=False)

    # Formatação bonita para o print
    format_dict = {
        'Retorno Total (Acum)': '{:.2%}',
        'Retorno Anual (CAGR)': '{:.2%}',
        'Volatilidade (a.a.)': '{:.2%}',
        'Sharpe (a.a.)': '{:.2f}'
    }
    print(df_resumo.style.format(format_dict).to_string())

    plt.show()
else:
    print("Nenhum modelo conseguiu otimizar com as restrições atuais no período de treino.")

In [ ]:
# @title Teste de Stress na Pandemia 2020
DATA_CORTE = '2020-03-11'  # Inicio da pandemia oficial
DATA_FINAL = '2022-05-22'  # Fim da pandemia oficial tambem
CAPITAL = 100000.00
META_ALVO_AA = 0.02
RISK_FREE_AA = 0.04 # Selic média estimada do período (2020-2022 foi baixa/média)

# Custos
CUSTO_FIXO, CUSTO_PROP, FATOR_CUSTO_FIXO =calcular_custos_dinamicos(CAPITAL)


print(f"--- CONFIGURAÇÃO DO BACKTEST ---")
print(f"Treino (Passado): Início dos dados até {DATA_CORTE}")
print(f"Teste (Simulação): {DATA_CORTE} até {DATA_FINAL}")


# 2. FATIAMENTO DOS DADOS
returns.index = pd.to_datetime(returns.index)

# A. Dados de Treino (O que o modelo sabe)
ret_train = returns.loc[:DATA_CORTE]

# B. Dados de Teste (O que aconteceu de fato na janela escolhida)
ret_test = returns.loc[DATA_CORTE:DATA_FINAL]

# Validação!
if ret_train.empty or ret_test.empty:
    raise ValueError("Erro: As datas selecionadas estão fora do alcance dos dados disponíveis.")

dias_teste = len(ret_test)
anos_no_teste = dias_teste / 252

print(f"Dias de Treino: {len(ret_train)}")
print(f"Dias de Teste:  {dias_teste} (~{anos_no_teste:.2f} anos)")

# --- criando os dados da janela de treino ---
mu_train = ret_train.mean().values
sigma_train = ret_train.cov().values
returns_np_train = ret_train.values
n = ret_train.shape[1] # número de ativos caso tenha mudado (certamente muda)

def run_markowitz_backtest():
    limites_marko = {
      'Financeiro': 0.2,
      'Energia': 0.18,
      'Materiais Básicos': 0.15,
      'Utilidade Pública': 0.15,
      'Tecnologia': 0.1,
      'Renda Fixa / Caixa': 0.5,
      'Criptoativos': 0.03,
      'Consumo Cíclico': 0.08,
      'Consumo Não-Cíclico': 0.12,
      'Saúde': 0.1,
      'Industrial': 0.1,
      'Imobiliário': 0.08,
      'Comunicações': 0.08,
      'Fundo Imobiliário': 0.15,
      'ETF': 0.12
    }
    # usando mu_train e sigma_train
    try:
        m = gp.Model("Markowitz_BT", env=env)
        m.Params.OutputFlag = 0
        x = m.addMVar(n, lb=0, ub=1, name="x")
        z = m.addMVar(n, vtype=GRB.BINARY, name="z")

        # Custos
        custos = (CUSTO_PROP * x.sum()) + (FATOR_CUSTO_FIXO * z.sum())

        # Retorno
        ret_ano = (mu_train * 252) @ x
        m.addConstr(ret_ano - custos >= META_ALVO_AA)

        # Risco (Minimizar) lambda = 0.2 para o moderado
        lambda_risk = 0.2
        sigma_train_anu = sigma_train * 252
        mu_train_anu = mu_train * 252

        m.setObjective((x @ (sigma_train_anu) @ x) - (lambda_risk * (mu_train_anu @ x)), GRB.MINIMIZE)
        # Restrições Padrão
        m.addConstr(x.sum() == 1)
        m.addConstr(z.sum() <= 25) # Cardinalidade
        m.addConstr(z.sum() >= 10)
        for i in range(n):
            m.addConstr(x[i] <= 0.50 * z[i]) # Máximos e mínimos de porcentagem de um ativo.
            m.addConstr(x[i] >= 0.05 * z[i])

        # retrição de Setores
        for setor, indices in indices_por_setor.items():
            if setor in limites_marko:
                m.addConstr(x[indices].sum() <= limites_marko[setor])

        m.optimize()
        return x.X if m.status == GRB.OPTIMAL else None
    except: return None

def run_cvar_backtest():
    # Usa returns_np_train

    limites_cvar = {
        'Financeiro': 0.10,
        'Energia': 0.10,
        'Materiais Básicos': 0.05,
        'Utilidade Pública': 0.15,
        'Tecnologia': 0.05,
        'Renda Fixa / Caixa': 0,
        'Criptoativos': 0.00,
        'Consumo Cíclico': 0.05,
        'Consumo Não-Cíclico': 0.10,
        'Saúde': 0.10,
        'Industrial': 0.05,
        'Imobiliário': 0.10,
        'Comunicações': 0.10,
        'Fundo Imobiliário': 0.20,
        'ETF': 0.20
    }

    try:
        T_t, N_t = returns_np_train.shape
        m = gp.Model("CVaR_BT", env=env)
        m.Params.OutputFlag = 0

        w = m.addVars(N_t, lb=0, ub=1)
        y = m.addVars(N_t, vtype=GRB.BINARY)
        eta = m.addVar(lb=-GRB.INFINITY)
        z_var = m.addVars(T_t, lb=0)

        # Obj CVaR
        alpha = 0.975
        m.setObjective(eta + (1/((1-alpha)*T_t)) * gp.quicksum(z_var[t] for t in range(T_t)), GRB.MINIMIZE)

        for t in range(T_t):
            r_port_t = gp.quicksum(w[i] * returns_np_train[t, i] for i in range(N_t))
            m.addConstr(z_var[t] >= -r_port_t - eta)

        m.addConstr(gp.quicksum(w[i] for i in range(N_t)) == 1)

        # Retorno
        ret_bruto = gp.quicksum(w[i] * mu_train[i] * 252 for i in range(N_t))
        custos = gp.quicksum(w[i] * CUSTO_PROP for i in range(N_t)) + gp.quicksum(y[i] * FATOR_CUSTO_FIXO for i in range(N_t))
        m.addConstr(ret_bruto - custos >= META_ALVO_AA)

        # Card/Setor
        m.addConstr(gp.quicksum(y[i] for i in range(N_t)) <= 25)
        m.addConstr(gp.quicksum(y[i] for i in range(N_t)) >= 17)
        for i in range(N_t):
            m.addConstr(w[i] <= 0.60 * y[i])
            m.addConstr(w[i] >= 0.04 * y[i])

        for setor, indices in indices_por_setor.items():
            if setor in limites_cvar:
                m.addConstr(gp.quicksum(w[i] for i in indices) <= limites_cvar[setor])

        m.optimize()
        return np.array([w[i].X for i in range(N_t)]) if m.status == GRB.OPTIMAL else None
    except: return None

def run_omega_backtest():
    # returns_np_train e Charnes-Cooper

    limites_omega = {
        'Financeiro': 0.30,
        'Energia': 0.35,
        'Materiais Básicos': 0.35,
        'Utilidade Pública': 0.15,
        'Tecnologia': 0.35,
        'Renda Fixa / Caixa': 0,
        'Criptoativos': 0.10,
        'Consumo Cíclico': 0.25,
        'Consumo Não-Cíclico': 0.10,
        'Saúde': 0.1,
        'Industrial': 0.25,
        'Imobiliário': 0.15,
        'Comunicações': 0.15,
        'Fundo Imobiliário': 0.20,
        'ETF': 0.20
    }

    try:
        T_t, N_t = returns_np_train.shape
        m = gp.Model("Omega_BT", env=env)
        m.Params.OutputFlag = 0

        y = m.addMVar(N_t, lb=0) # y = w * tau
        z = m.addMVar(N_t, vtype=GRB.BINARY)
        tau = m.addVar(lb=0)
        nu = m.addMVar(T_t, lb=0)
        delta = m.addMVar(T_t, lb=0)
        z_tau = m.addMVar(N_t, lb=0) # Linearização z*tau


        SELIC_2020 = 0.0875 #média da selic no recorte
        L_diario = (1 + SELIC_2020)**(1/252) - 1
        #L_diario é Threshold (aprox CDI diário)

        m.setObjective(nu.sum(), GRB.MAXIMIZE)

        # Linearização z*tau
        M_val = 100
        for i in range(N_t):
            m.addConstr(z_tau[i] <= M_val * z[i])
            m.addConstr(z_tau[i] <= tau)
            m.addConstr(z_tau[i] >= tau - M_val*(1-z[i]))

        # Omega Core
        m.addConstr(delta.sum() == 1)
        m.addConstr(y.sum() == tau)

        # Retorno com custos
        ret_ano_t = gp.quicksum(y[i] * mu_train[i] * 252 for i in range(N_t))
        custos_t = gp.quicksum(y[i] * CUSTO_PROP for i in range(N_t)) + gp.quicksum(z_tau[i] * FATOR_CUSTO_FIXO for i in range(N_t))
        m.addConstr(ret_ano_t - custos_t >= META_ALVO_AA * tau)

        # Upside/Downside do omega ratio
        for t in range(T_t):
            # y @ r_t
            val_t = gp.quicksum(y[i] * returns_np_train[t, i] for i in range(N_t))
            m.addConstr(val_t - L_diario * tau == nu[t] - delta[t])

        # Card e Setorizacao
        m.addConstr(z.sum() <= 13)
        m.addConstr(z.sum() >= 8)
        for i in range(N_t):
            m.addConstr(y[i] <= 0.20 * z_tau[i]) # usando z_tau para consistência
            m.addConstr(y[i] >= 0.03 * z_tau[i])
        for setor, indices in indices_por_setor.items():
            if setor in limites_omega:
                m.addConstr(y[indices].sum() <= limites_omega[setor] * tau)

        m.optimize()
        if m.status == GRB.OPTIMAL and tau.X > 1e-6:
            return y.X / tau.X
        return None
    except: return None

# 3. resultados (Baseada no passado)
resultados = {}

print("\n1. Otimizando Markowitz (Pré-Crise)...")
w_mk = run_markowitz_backtest()
if w_mk is not None: resultados['Markowitz'] = w_mk

print("2. Otimizando CVaR (Pré-Crise)...")
w_cv = run_cvar_backtest()
if w_cv is not None: resultados['CVaR'] = w_cv

print("3. Otimizando Omega (Pré-Crise)...")
w_om = run_omega_backtest()
if w_om is not None: resultados['Omega'] = w_om

# Benchmark 1/N só pra servir de referência se foi bom ou ruim.
w_ew = np.ones(n) / n
resultados['Benchmark (1/N)'] = w_ew

# 4. SIMULAÇÃO E RESULTADOS ANUALIZADOS

if len(resultados) > 0:
    plt.figure(figsize=(14, 7))

    df_performance = pd.DataFrame(index=ret_test.index)
    resumo_final = []

    for nome, pesos in resultados.items():
        if len(pesos) == ret_test.shape[1]:
            # 1. Retorno Diário na Janela de Teste
            daily_ret = ret_test.dot(pesos)

            # 2. Curva de Patrimônio
            cumulative = (1 + daily_ret).cumprod() * 100
            df_performance[nome] = cumulative

            # Plot
            plt.plot(cumulative, label=nome, linewidth=2 if 'Benchmark' not in nome else 1.5,
                     linestyle='--' if 'Benchmark' in nome else '-')

            # 3. Métricas (Considerando a janela específica)
            ret_total_abs = (cumulative.iloc[-1] / 100) - 1

            # CAGR (Anualizado)
            if anos_no_teste > 0:
                cagr = (1 + ret_total_abs) ** (1 / anos_no_teste) - 1
            else:
                cagr = 0

            # Volatilidade (Anualizada)
            vol_anual = daily_ret.std() * np.sqrt(252)

            # Sharpe
            sharpe = (cagr - RISK_FREE_AA) / vol_anual if vol_anual > 0 else 0

            resumo_final.append({
                'Modelo': nome,
                'Retorno Total (Janela)': ret_total_abs,
                'Retorno Anual (CAGR)': cagr,
                'Volatilidade (a.a.)': vol_anual,
                'Sharpe (a.a.)': sharpe
            })

    # Gráfico
    plt.title(f'Simulação de Carteira: {DATA_CORTE} a {DATA_FINAL}', fontsize=16)
    plt.ylabel('Evolução (Base 100)')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Linha vertical para marcar o início (visual)
    plt.axvline(pd.to_datetime(DATA_CORTE), color='red', linestyle=':', alpha=0.5, label='Início Teste')

    print("\n=== RESULTADO DA JANELA ESPECÍFICA ===")
    df_resumo = pd.DataFrame(resumo_final).set_index('Modelo')
    df_resumo = df_resumo.sort_values('Retorno Anual (CAGR)', ascending=False)

    # Formatação
    format_dict = {
        'Retorno Total (Janela)': '{:.2%}',
        'Retorno Anual (CAGR)': '{:.2%}',
        'Volatilidade (a.a.)': '{:.2%}',
        'Sharpe (a.a.)': '{:.2f}'
    }
    print(df_resumo.style.format(format_dict).to_string())

    plt.show()

else:
    print("Erro: Modelos não convergiram com os dados de treino fornecidos.")

In [ ]:
# @title código para saber quantos dias tem de informação
print("=== INSPEÇÃO DOS DADOS ===")
print(f"Data Inicial disponível: {returns.index.min()}")
print(f"Data Final disponível:   {returns.index.max()}")
print(f"Total de dias úteis:     {len(returns)}")

# Discussão

## Limitações
* **Dependência de dados históricos:** Todos os modelos dependem da qualidade e representatividade dos retornos históricos ($\Sigma$ e $r_{t,i}$), além de pressupor uma razoável estacionariedade, o que pode não ocorrer em mercados voláteis.
* **Turnover e custos reais:** A versão atual considera custos (taxa proporcional e custo fixo aproximado), mas não modela turnover dinâmico multi-período nem efeitos fiscais detalhados, ou seja, o modelo consegue criar uma carteira de investimentos do zero, mas não consegue atualizar uma carteira já existente.
* **Liquidez e impacto de mercado:** Não há modelagem de *slippage* ou limites de liquidez para grandes posições.
* **Simplificações nas restrições:** A cardinalidade e os limites setoriais são discretos e simplificados, e podem exigir ajuste conforme o universo real de ativos.

## Casos Especiais
A aversão ao risco $\lambda$ no Markowitz controla a transição entre portfólios de mínima variância ($\lambda \to 0$) e portfólios focalizados em retorno ($\lambda$ grande). Ajustando $\lambda$ obtemos um espectro do conservador ao agressivo.

## Generalizações
* **Extensão a multi-período:** (*dynamic programming / model predictive control*) para capturar turnover, reinvestimento e taxas de transação ao longo do tempo.
* **Inclusão de classes adicionais:** (renda fixa, derivativos, cripto) exigiria ajustes em $\Sigma$ e em restrições de liquidez/setoriais.

## Variações / Melhorias
* Trocar matriz de covariância por modelos robustos (estimadores *shrinkage*) ou usar métricas alternativas como Mean Absolute Deviation (MAD) para linearizar Markowitz.
* Para o Omega, testar Dinkelbach vs Charnes-Cooper em instâncias pequenas para comparar convergência e tempo de solver (no código, Charnes-Cooper mostrou-se bem mais rápido).

## Trabalhos Futuros
Possibilitar que receba uma carteira de investimento de input e a partir dela atualizá-la e fazer novos investimentos, para isso teríamos que usar o turnover. Além disso, utilizar e estudar novos modelos de cada perfil para convergir com os desejos do usuário e para maior variabilidade de escolhas de carteira de investimento.